# EDA 
**Tujuan:** Memahami struktur data, kualitas data, dan kelayakan konstruksi network sebelum membangun SNA model.

**Alur EDA:**
1. Data Overview & Quality Check
2. Username & Post Distribution Analysis
3. Temporal Analysis
4. Interaction Metrics Analysis
5. Content Analysis (Mention & Hashtag Extraction)
6. Sentiment Analysis
7. Keyword Analysis
8. Network Feasibility Estimation
9. ✅ Kesimpulan & Rekomendasi untuk SNA

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import re
import warnings
from collections import Counter
from datetime import datetime
import json

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ Libraries loaded')

In [ ]:
# ── Load Dataset ─────────────────────────────────────────────────────────────
# Ganti path sesuai lokasi dataset kamu
DATA_PATH = '../data/raw/dataset.csv'

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Shape: {df.shape}')
df.head(3)

---
## 1. Data Overview & Quality Check

In [ ]:
# ── Basic Info ────────────────────────────────────────────────────────────────
print('=' * 60)
print(f'Rows    : {df.shape[0]:,}')
print(f'Columns : {df.shape[1]}')
print('=' * 60)
print('\nDtypes:')
print(df.dtypes)
print('\nMemory usage:', df.memory_usage(deep=True).sum() / 1024**2, 'MB')

In [ ]:
# ── Missing Values Analysis ───────────────────────────────────────────────────
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('missing_pct', ascending=False)

print('\n📋 Missing Values Report:')
print(missing[missing.missing_count > 0].to_string())

fig, ax = plt.subplots(figsize=(10, 5))
missing_filtered = missing[missing.missing_count > 0]
bars = ax.barh(missing_filtered.index, missing_filtered.missing_pct, color='salmon')
ax.set_xlabel('Missing %')
ax.set_title('Missing Values per Column')
for bar, val in zip(bars, missing_filtered.missing_pct):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('../data/processed/eda_missing_values.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Duplicates Check ─────────────────────────────────────────────────────────
dup_url = df['url'].duplicated().sum() if 'url' in df.columns else 'N/A'
dup_ext = df['external_id'].duplicated().sum() if 'external_id' in df.columns else 'N/A'
dup_full = df.duplicated().sum()

print(f'Duplicate rows (full)    : {dup_full}')
print(f'Duplicate URLs           : {dup_url}')
print(f'Duplicate external_id    : {dup_ext}')

In [ ]:
# ── Source Type Distribution (Filter Instagram only) ────────────────────────
print('source_type distribution:')
print(df['source_type'].value_counts())

# Filter hanya Instagram jika ada multiple source types
if df['source_type'].nunique() > 1:
    print('\n⚠️  Dataset contains multiple source types, filtering Instagram...')
    ig_keywords = ['instagram', 'ig']
    mask = df['source_type'].str.lower().str.contains('|'.join(ig_keywords), na=False)
    df_ig = df[mask].copy()
    print(f'After filter: {df_ig.shape[0]:,} rows')
else:
    df_ig = df.copy()
    print('\n✅ All data is Instagram')

---
## 2. Username & Post Distribution

In [ ]:
# ── Username Stats ────────────────────────────────────────────────────────────
df_ig['username'] = df_ig['username'].str.strip().str.lower()
df_ig = df_ig[df_ig['username'].notna() & (df_ig['username'] != '')]

n_users = df_ig['username'].nunique()
n_posts = len(df_ig)
posts_per_user = df_ig.groupby('username').size()

print(f'Unique usernames : {n_users:,}')
print(f'Total posts      : {n_posts:,}')
print(f'Avg posts/user   : {posts_per_user.mean():.2f}')
print(f'Max posts/user   : {posts_per_user.max()}')
print(f'Median posts/user: {posts_per_user.median()}')
print(f'\nTop 20 most active users:')
print(posts_per_user.sort_values(ascending=False).head(20).to_string())

In [ ]:
# ── Post Frequency Distribution (Power Law Check) ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
axes[0].hist(posts_per_user, bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Posts per User')
axes[0].set_ylabel('Count')
axes[0].set_title('Post Distribution (Linear)')

# Log-log scale (Power law check)
counts = Counter(posts_per_user)
x = list(counts.keys())
y = list(counts.values())
axes[1].scatter(x, y, alpha=0.6, color='steelblue', s=20)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('Posts per User (log)')
axes[1].set_ylabel('Frequency (log)')
axes[1].set_title('Post Distribution (Log-Log) — Power Law Check')

plt.tight_layout()
plt.savefig('../data/processed/eda_post_distribution.png', bbox_inches='tight')
plt.show()

# Kalau log-log roughly linear → power law → typical social network behavior ✅

---
## 3. Temporal Analysis

In [ ]:
# ── Parse Date ────────────────────────────────────────────────────────────────
df_ig['date'] = pd.to_datetime(df_ig['date'], errors='coerce')
print(f'Date range: {df_ig["date"].min()} → {df_ig["date"].max()}')
print(f'Null dates: {df_ig["date"].isna().sum()}')

df_ig = df_ig.dropna(subset=['date'])
df_ig['year_month'] = df_ig['date'].dt.to_period('M')
df_ig['day_of_week'] = df_ig['date'].dt.day_name()
df_ig['hour'] = df_ig['date'].dt.hour

In [ ]:
# ── Posts Over Time ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Monthly trend
monthly = df_ig.groupby('year_month').size()
monthly.plot(ax=axes[0,0], color='steelblue', marker='o', ms=4)
axes[0,0].set_title('Posts per Month')
axes[0,0].set_xlabel('')
axes[0,0].tick_params(axis='x', rotation=45)

# Day of week
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = df_ig['day_of_week'].value_counts().reindex(dow_order)
dow.plot(kind='bar', ax=axes[0,1], color='steelblue', edgecolor='white')
axes[0,1].set_title('Posts by Day of Week')
axes[0,1].tick_params(axis='x', rotation=45)

# Hour of day (if time data available)
hour_counts = df_ig['hour'].value_counts().sort_index()
hour_counts.plot(kind='bar', ax=axes[1,0], color='coral', edgecolor='white')
axes[1,0].set_title('Posts by Hour of Day')
axes[1,0].set_xlabel('Hour')

# Active users over time (unique usernames per month)
monthly_users = df_ig.groupby('year_month')['username'].nunique()
monthly_users.plot(ax=axes[1,1], color='green', marker='s', ms=4)
axes[1,1].set_title('Unique Active Users per Month')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../data/processed/eda_temporal.png', bbox_inches='tight')
plt.show()

---
## 4. Interaction Metrics Analysis

In [ ]:
# ── Numeric Stats ─────────────────────────────────────────────────────────────
interaction_cols = ['total_like', 'total_share', 'total_view', 'total_interaction']
existing_cols = [c for c in interaction_cols if c in df_ig.columns]

print('📊 Interaction Metrics Summary:')
print(df_ig[existing_cols].describe(percentiles=[.25, .5, .75, .9, .95, .99]))

# Check correlation
print('\n📊 Correlation Matrix:')
print(df_ig[existing_cols].corr().round(3))

In [ ]:
# ── Interaction Distribution (Log Scale) ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple']

for ax, col, color in zip(axes.flatten(), existing_cols, colors):
    data = df_ig[col].dropna()
    data_log = np.log1p(data)
    ax.hist(data_log, bins=50, color=color, edgecolor='white', alpha=0.8)
    ax.set_title(f'{col} (log1p scale)')
    ax.set_xlabel('log1p(value)')
    ax.axvline(data_log.median(), color='red', linestyle='--', label=f'median={data.median():.0f}')
    ax.legend(fontsize=8)

plt.suptitle('Interaction Metrics Distribution', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/eda_interactions.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Top Users by Interaction ──────────────────────────────────────────────────
user_metrics = df_ig.groupby('username').agg(
    post_count=('username', 'count'),
    total_interaction=('total_interaction', 'sum'),
    avg_interaction=('total_interaction', 'mean'),
    total_like=('total_like', 'sum'),
    total_view=('total_view', 'sum'),
    total_share=('total_share', 'sum'),
).reset_index().sort_values('total_interaction', ascending=False)

print('Top 20 users by total interaction:')
print(user_metrics.head(20).to_string(index=False))

# Save for later use in SNA
user_metrics.to_csv('../data/processed/user_metrics.csv', index=False)
print('\n✅ Saved to data/processed/user_metrics.csv')

In [ ]:
# ── Interaction Coverage Check ────────────────────────────────────────────────
null_interaction = df_ig['total_interaction'].isna().sum()
total = len(df_ig)
coverage = (total - null_interaction) / total * 100

print(f'total_interaction coverage: {coverage:.1f}% ({total-null_interaction:,}/{total:,} rows)')

# Posts with 0 interaction
zero_interaction = (df_ig['total_interaction'] == 0).sum()
print(f'Posts with 0 interaction : {zero_interaction:,}')

# INSIGHT: Check if missing interaction correlates with certain users
missing_by_user = df_ig[df_ig['total_interaction'].isna()]['username'].value_counts()
print(f'\nUsers with most missing interactions (top 10):')
print(missing_by_user.head(10))

---
## 5. Content Analysis — Mention & Hashtag Extraction
> ⚠️ **PALING KRITIS** untuk SNA: mentions di content = edges antar nodes

In [ ]:
# ── Extract @mentions dan #hashtags dari content ──────────────────────────────
def extract_mentions(text):
    if pd.isna(text):
        return []
    return re.findall(r'@([A-Za-z0-9_.]+)', str(text))

def extract_hashtags(text):
    if pd.isna(text):
        return []
    return re.findall(r'#([A-Za-z0-9_]+)', str(text))

# Apply to both content & content_processed
content_col = 'content_processed' if 'content_processed' in df_ig.columns else 'content'
print(f'Using content column: {content_col}')

df_ig['mentions'] = df_ig[content_col].apply(extract_mentions)
df_ig['hashtags'] = df_ig[content_col].apply(extract_hashtags)
df_ig['n_mentions'] = df_ig['mentions'].apply(len)
df_ig['n_hashtags'] = df_ig['hashtags'].apply(len)

In [ ]:
# ── Mention Statistics ────────────────────────────────────────────────────────
posts_with_mentions = (df_ig['n_mentions'] > 0).sum()
total_mentions = df_ig['n_mentions'].sum()

print(f'Posts with ≥1 mention  : {posts_with_mentions:,} ({posts_with_mentions/len(df_ig)*100:.1f}%)')
print(f'Total mentions found   : {total_mentions:,}')
print(f'Avg mentions per post  : {total_mentions/len(df_ig):.2f}')
print(f'Max mentions in 1 post : {df_ig["n_mentions"].max()}')

# All mentioned usernames
all_mentions = [m.lower() for mentions in df_ig['mentions'] for m in mentions]
mention_counter = Counter(all_mentions)

print(f'\nUnique mentioned usernames: {len(mention_counter):,}')
print('\nTop 30 most mentioned users:')
for user, count in mention_counter.most_common(30):
    print(f'  @{user}: {count}')

In [ ]:
# ── Build Edge List from Mentions ─────────────────────────────────────────────
# Edge: poster → mentioned_user (directed)
edges_mention = []
for _, row in df_ig.iterrows():
    poster = row['username']
    for mentioned in row['mentions']:
        mentioned = mentioned.lower().strip()
        if mentioned and mentioned != poster:
            edges_mention.append({
                'source': poster,
                'target': mentioned,
                'date': str(row['date']),
                'type': 'mention',
                'post_interaction': row.get('total_interaction', 0)
            })

edges_df = pd.DataFrame(edges_mention)
print(f'Total mention-based edges: {len(edges_df):,}')

if len(edges_df) > 0:
    # Aggregate: weight = frequency of A mentioning B
    edges_agg = edges_df.groupby(['source', 'target']).agg(
        weight=('type', 'count'),
        avg_interaction=('post_interaction', 'mean')
    ).reset_index()
    print(f'Unique directed edges    : {len(edges_agg):,}')
    print(f'\nTop 20 strongest edges:')
    print(edges_agg.sort_values('weight', ascending=False).head(20).to_string(index=False))
    
    edges_agg.to_csv('../data/processed/edges_mention.csv', index=False)
    print('\n✅ Saved to data/processed/edges_mention.csv')
else:
    print('\n⚠️  NO MENTIONS FOUND in content!')
    print('→ SNA will need to use co-keyword network as primary edge source')
    print('→ See Section 8: Network Feasibility')

In [ ]:
# ── Hashtag Analysis ──────────────────────────────────────────────────────────
all_hashtags = [h.lower() for tags in df_ig['hashtags'] for h in tags]
hashtag_counter = Counter(all_hashtags)

print(f'Posts with ≥1 hashtag  : {(df_ig["n_hashtags"] > 0).sum():,}')
print(f'Unique hashtags        : {len(hashtag_counter):,}')
print('\nTop 30 hashtags:')
for tag, count in hashtag_counter.most_common(30):
    print(f'  #{tag}: {count}')

# Save hashtag co-occurrence potential
df_ig['hashtags_str'] = df_ig['hashtags'].apply(lambda x: '|'.join([h.lower() for h in x]))
df_ig[['username', 'date', 'hashtags_str', 'mentions']].to_csv(
    '../data/processed/posts_with_meta.csv', index=False)
print('\n✅ Saved posts_with_meta.csv')

---
## 6. Sentiment Analysis

In [ ]:
# ── Sentiment Distribution ────────────────────────────────────────────────────
if 'sentiment_label' in df_ig.columns:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Overall distribution
    sent_counts = df_ig['sentiment_label'].value_counts()
    colors_sent = {'positive': 'mediumseagreen', 'negative': 'salmon', 'neutral': 'steelblue'}
    bar_colors = [colors_sent.get(s.lower(), 'gray') for s in sent_counts.index]
    axes[0].bar(sent_counts.index, sent_counts.values, color=bar_colors, edgecolor='white')
    axes[0].set_title('Overall Sentiment Distribution')
    axes[0].set_ylabel('Count')
    
    # Sentiment score distribution
    if 'sentiment_score' in df_ig.columns:
        df_ig['sentiment_score'].hist(bins=50, ax=axes[1], color='steelblue', edgecolor='white')
        axes[1].set_title('Sentiment Score Distribution')
        axes[1].set_xlabel('Score')
    
    # Sentiment over time
    sent_time = df_ig.groupby(['year_month', 'sentiment_label']).size().unstack(fill_value=0)
    sent_time.plot(ax=axes[2], colormap='RdYlGn')
    axes[2].set_title('Sentiment Over Time')
    axes[2].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.savefig('../data/processed/eda_sentiment.png', bbox_inches='tight')
    plt.show()

    # Top users by negative sentiment (brand risk)
    neg_users = df_ig[df_ig['sentiment_label'].str.lower()=='negative'].groupby('username').size()
    print('Top 10 users with most negative posts:')
    print(neg_users.sort_values(ascending=False).head(10))
    
    # Sentiment per keyword
    if 'keyword' in df_ig.columns:
        print('\nSentiment breakdown per keyword:')
        print(df_ig.groupby(['keyword', 'sentiment_label']).size().unstack(fill_value=0))

---
## 7. Keyword Analysis

In [ ]:
# ── Keyword Distribution ──────────────────────────────────────────────────────
if 'keyword' in df_ig.columns:
    kw_counts = df_ig['keyword'].value_counts()
    print(f'Unique keywords: {len(kw_counts)}')
    print('\nKeyword distribution:')
    print(kw_counts.to_string())
    
    fig, ax = plt.subplots(figsize=(10, 5))
    kw_counts.head(20).plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title('Posts per Keyword')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig('../data/processed/eda_keywords.png', bbox_inches='tight')
    plt.show()
    
    # Users posting multiple keywords = bridge nodes in SNA
    user_kw = df_ig.groupby('username')['keyword'].nunique()
    multi_kw_users = user_kw[user_kw > 1]
    print(f'\nUsers posting on multiple keywords (potential bridge nodes): {len(multi_kw_users):,}')
    print(multi_kw_users.sort_values(ascending=False).head(20))

---
## 8. Network Feasibility Estimation

In [ ]:
# ── Network Feasibility Summary ───────────────────────────────────────────────
print('=' * 60)
print('NETWORK FEASIBILITY REPORT')
print('=' * 60)

# Method 1: Mention-based edges
n_mention_edges = len(edges_df) if 'edges_df' in dir() and len(edges_df) > 0 else 0
print(f'\n[Method 1] Mention-Based Edges: {n_mention_edges:,}')
if n_mention_edges > 0:
    mentioned_users_in_dataset = set(edges_df['target']).intersection(set(df_ig['username']))
    print(f'  Targets found in dataset: {len(mentioned_users_in_dataset):,}')
    print(f'  External targets (not in dataset): {len(set(edges_df["target"])) - len(mentioned_users_in_dataset):,}')

# Method 2: Co-keyword edges (same keyword, same timeframe)
if 'keyword' in df_ig.columns:
    kw_groups = df_ig.groupby('keyword')['username'].nunique()
    potential_kw_edges = sum(n*(n-1)//2 for n in kw_groups)
    print(f'\n[Method 2] Co-Keyword Potential Edges: {potential_kw_edges:,}')
    print(f'  (estimated, before time-windowing)')

# Method 3: Hashtag co-occurrence edges
posts_with_hashtags = (df_ig['n_hashtags'] > 0).sum()
print(f'\n[Method 3] Hashtag Co-occurrence:')
print(f'  Posts with hashtags: {posts_with_hashtags:,}')
print(f'  Unique hashtags    : {len(hashtag_counter):,}')

print('\n' + '=' * 60)
print('NODES')
print(f'  Total unique usernames  : {df_ig["username"].nunique():,}')
print('\nRECOMMENDED EDGE STRATEGY:')
if n_mention_edges > 100:
    print('  ✅ Mention edges are sufficient for primary SNA')
    print('  ➕ Supplement with co-keyword edges for denser graph')
else:
    print('  ⚠️  Mention edges are sparse (<100)')
    print('  → PRIMARY: Use co-keyword + co-hashtag edges')
    print('  → WEIGHT edges by interaction score')
print('=' * 60)

In [ ]:
# ── Build Co-Keyword Edges (time-windowed) ─────────────────────────────────────
# Logic: if user A and user B post on same keyword within 7 days → edge
# Weight = interaction score of posts

if 'keyword' in df_ig.columns:
    TIME_WINDOW_DAYS = 7
    edges_cokw = []
    
    for keyword, group in df_ig.groupby('keyword'):
        group_sorted = group.sort_values('date')
        users = group_sorted[['username', 'date', 'total_interaction']].values
        
        for i in range(len(users)):
            for j in range(i+1, len(users)):
                u1, d1, w1 = users[i]
                u2, d2, w2 = users[j]
                
                if u1 == u2:
                    continue
                
                # Time window check
                if abs((pd.Timestamp(d2) - pd.Timestamp(d1)).days) <= TIME_WINDOW_DAYS:
                    weight = ((w1 or 0) + (w2 or 0)) / 2
                    edges_cokw.append({
                        'source': u1, 'target': u2,
                        'keyword': keyword, 'weight': weight, 'type': 'co_keyword'
                    })
                else:
                    break  # sorted by date, no need to continue
    
    if edges_cokw:
        edges_cokw_df = pd.DataFrame(edges_cokw)
        edges_cokw_agg = edges_cokw_df.groupby(['source','target']).agg(
            weight=('weight', 'mean'),
            co_keyword_count=('keyword', 'count')
        ).reset_index()
        print(f'Co-keyword edges (time-windowed): {len(edges_cokw_agg):,}')
        edges_cokw_agg.to_csv('../data/processed/edges_cokeyword.csv', index=False)
        print('✅ Saved edges_cokeyword.csv')
    else:
        print('No co-keyword edges found within time window')

In [ ]:
# ── Save clean dataset ────────────────────────────────────────────────────────
cols_to_keep = [
    'date', 'keyword', 'username', 'content', 'content_processed',
    'total_like', 'total_share', 'total_view', 'total_interaction',
    'sentiment_label', 'sentiment_score',
    'mentions', 'hashtags', 'n_mentions', 'n_hashtags', 'url', 'external_id'
]
cols_to_keep = [c for c in cols_to_keep if c in df_ig.columns]
df_clean = df_ig[cols_to_keep].copy()

# Convert lists to strings for CSV storage
df_clean['mentions'] = df_clean['mentions'].apply(lambda x: '|'.join(x) if isinstance(x, list) else '')
df_clean['hashtags'] = df_clean['hashtags'].apply(lambda x: '|'.join(x) if isinstance(x, list) else '')

df_clean.to_csv('../data/processed/dataset_clean.csv', index=False)
print(f'✅ Clean dataset saved: {df_clean.shape}')

---
## ✅ 9. EDA Summary & Rekomendasi untuk SNA

> Jalankan cell ini setelah semua cell di atas selesai

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print('''
╔══════════════════════════════════════════════════════════════╗
║               EDA SUMMARY — INSTAGRAM SNA                   ║
╚══════════════════════════════════════════════════════════════╝
''')

print(f'  Total posts      : {len(df_ig):,}')
print(f'  Unique users     : {df_ig["username"].nunique():,}')
print(f'  Date range       : {df_ig["date"].min().date()} → {df_ig["date"].max().date()}')
print(f'  Keywords         : {df_ig["keyword"].nunique() if "keyword" in df_ig.columns else "N/A"}')
print(f'  Mention edges    : {n_mention_edges:,}')
print(f'  Interaction cov. : {(df_ig["total_interaction"].notna().sum() / len(df_ig) * 100):.1f}%')

print('''
──────────────────────────────────────────────────────────────
  FILES GENERATED:
  • data/processed/dataset_clean.csv     → Clean dataset
  • data/processed/user_metrics.csv      → Per-user aggregates
  • data/processed/edges_mention.csv     → Mention-based edges
  • data/processed/edges_cokeyword.csv   → Co-keyword edges
  • data/processed/posts_with_meta.csv   → Posts + mentions/hashtags
──────────────────────────────────────────────────────────────
  NEXT STEP: Run notebook 02_SNA_Analysis.ipynb
──────────────────────────────────────────────────────────────
''')